# F-09 Whisper 파인튜닝 — Colab Pro+ (A100)

**실행 전 체크리스트**
- [ ] 런타임 유형: A100 GPU 선택
- [ ] 런타임 > 백그라운드 실행 활성화
- [ ] Google Drive 마운트 확인

**실행 순서**: 셀 01 → 02 → 03 → 04 → 05 → 06 순서대로 실행

In [ ]:
# 셀 01 — 라이브러리 설치
!pip install -q \
    transformers==4.44.0 \
    datasets==2.21.0 \
    peft==0.12.0 \
    accelerate==0.34.0 \
    evaluate==0.4.3 \
    jiwer==3.0.4 \
    librosa==0.10.2 \
    soundfile==0.12.1 \
    tensorboard

In [ ]:
# 셀 02 — Google Drive 마운트 및 경로 설정
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT     = Path('/content/drive/MyDrive/Dadam_dataSet')
DATASET_PATH   = DRIVE_ROOT / 'processed/senior_speech'  # preprocess.py 출력 경로
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints/whisper-senior'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print('Drive 마운트 완료')
print(f'데이터셋 경로: {DATASET_PATH}')

In [ ]:
# 셀 03 — 전처리 실행 (preprocess.py 아직 안 돌렸으면 여기서 실행)
import shutil

shutil.copy('/content/drive/MyDrive/Dadam/whisper/preprocess.py', '/content/preprocess.py')
!python /content/preprocess.py

print('전처리 완료')

In [ ]:
# 셀 04 — 모델·프로세서 로드 및 LoRA 설정
import torch
from datasets import load_from_disk
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID = 'openai/whisper-large-v3-turbo'  # large-v3 증류 모델 — 속도 6배, 한국어 성능 근접

# 프로세서: 오디오 → log-mel 스펙트로그램 + 토크나이저
processor = WhisperProcessor.from_pretrained(MODEL_ID, language='Korean', task='transcribe')

# 베이스 모델 로드
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language='Korean', task='transcribe')
model.config.suppress_tokens = []

# LoRA 설정 — turbo는 디코더 4층이므로 Q·V·K·Out 전체 적용해 학습 효율 확보
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=32,
    lora_alpha=64,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'out_proj'],
    lora_dropout=0.05,
    bias='none',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# 데이터셋 로드
dataset = load_from_disk(str(DATASET_PATH))
print(dataset)

In [ ]:
# 셀 05 — DataCollator 및 Trainer 설정
import numpy as np
from dataclasses import dataclass
from typing import Any
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
import evaluate

cer_metric = evaluate.load('cer')  # 한국어는 CER(음절 단위)이 WER보다 적합


@dataclass
class WhisperDataCollator:
    """배치별 패딩 처리 — 가변 길이 오디오를 30초로 패딩"""
    processor: Any

    def __call__(self, features):
        audio_arrays = [f['audio']['array'] for f in features]
        texts = [f['text'] for f in features]

        # log-mel 스펙트로그램 변환
        inputs = self.processor(
            audio_arrays,
            sampling_rate=16_000,
            return_tensors='pt',
            padding=True,
        )

        # 텍스트 → 토큰 ID (레이블)
        labels = self.processor.tokenizer(
            texts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=448,
        ).input_ids

        # 패딩 토큰을 -100으로 마스킹 (loss 계산 제외)
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {
            'input_features': inputs.input_features,
            'labels': labels,
        }


def compute_metrics(pred):
    """CER 계산 — 검증 중 호출"""
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {'cer': round(cer, 4)}


training_args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    per_device_train_batch_size=32,        # turbo는 VRAM 여유로 16→32 증가
    gradient_accumulation_steps=2,         # 유효 배치: 64
    learning_rate=1e-4,
    warmup_steps=500,
    max_steps=4000,
    gradient_checkpointing=True,           # VRAM 절감 (속도 일부 희생)
    fp16=True,
    evaluation_strategy='steps',
    eval_steps=500,
    save_strategy='steps',
    save_steps=500,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,               # CER은 낮을수록 좋음
    predict_with_generate=True,
    generation_max_length=448,
    report_to='tensorboard',
    save_total_limit=3,                    # 체크포인트 최대 3개 유지
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    data_collator=WhisperDataCollator(processor=processor),
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

print('Trainer 설정 완료')

In [ ]:
# 셀 06 — 학습 시작
# 백그라운드 실행 활성화 후 실행할 것 (런타임 > 백그라운드 실행)
trainer.train()

# 최종 어댑터 저장 (LoRA 가중치만, 수 MB)
model.save_pretrained(str(CHECKPOINT_DIR / 'final'))
processor.save_pretrained(str(CHECKPOINT_DIR / 'final'))
print(f'학습 완료. 저장 위치: {CHECKPOINT_DIR}/final')